# Census table download

Downloads ACS 5-year table for every Hawaiʻi block group from Census API.

Needs two settings in the `.env` file in the repo's top folder: `CENSUS_API_KEY` (your Census API key) and `SCOVI_PROJECT_DIR` (the Nextcloud "SCOVI Project" folder, whose `census/JPL` raw census files the downloads are compared with).

In [ ]:
import csv
import glob
import json
import os
import urllib.parse
import urllib.request

from dotenv import load_dotenv

load_dotenv()

# Tables and ACS 5-year release to download, each with the JPL folder it belongs in
TABLES = [
    "B25034",  # age_of_structure
    "B27010",  # health_insurance
    "B28010",  # households_w_computer
    "B28011",  # internet_subscription
    "C16002",  # limited_english_speaking
    "B09019",  # living_arrangements
    "C17002",  # income_share_of_fpl
    "B01001",  # person_under_5_65
    "B02001",  # race_origin
    "B25003",  # tenure
    "B25014",  # tenure_by_occupants_per_room
    "B11003",  # family_type_by_children
    "B25044",  # tenure_by_vehicles
    "B23025",  # employment_status
    "B25070",  # renter_cost_burden
    "B25091",  # owner_cost_burden
    # "B25046", # aggregate_vehicles: no longer used
    # "B19001", # household_income: no longer used
]
YEAR = 2024

OUTPUT_DIR = os.path.expanduser("~/Downloads")

# The raw census files path to Nextcloud, only use for comparison, must be manually deposited there after download
RAW_DIR = os.path.join(os.path.expanduser(os.environ["SCOVI_PROJECT_DIR"]), "Metrics/All Exposures/census/JPL")

# Free key from api.census.gov/data/key_signup.html
API_KEY = os.environ["CENSUS_API_KEY"]

HAWAII_FIPS = "15"

In [ ]:
# One request to the ACS 5-year API, as parsed JSON
def census_api_get(year, path, params):
    query = urllib.parse.urlencode(params + [("key", API_KEY)])
    with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs5{path}?{query}") as response:
        return json.load(response)


# A cell as data.census.gov shows it: the API's note ("-", "**", "(X)"...) where it has one, 
# "null" where there is no value, else the value
def website_cell(value, note):
    if note is not None:
        return note
    if value is None:
        return "null"
    return str(value)


# A table for every Hawaiʻi block group, laid out like a website download: the code row, the label row, 
# then one row per block group in place id order
def download_table(table, year):
    variables = census_api_get(year, f"/groups/{table}.json", [])["variables"]
    response = census_api_get(year, "", [
        ("get", f"group({table})"),
        ("for", "block group:*"),
        ("in", f"state:{HAWAII_FIPS}"),
        ("in", "county:*"),
        ("in", "tract:*"),
    ])
    records = sorted((dict(zip(response[0], row)) for row in response[1:]), key=lambda record: record["GEO_ID"])

    estimates = sorted(code for code in variables if code.startswith(f"{table}_") and code.endswith("E"))
    codes = ["GEO_ID", "NAME"] + [code for estimate in estimates for code in (estimate, estimate[:-1] + "M")]
    labels = [variables[code]["label"] for code in codes]
    rows = [[website_cell(record[code], record.get(code + "A")) for code in codes] for record in records]
    return [codes, labels] + rows


# Saves the rows as ACSDT5Y<year>.<table>-Data.csv, the name the pipeline reads the table and years from
def save_census_csv(rows, table, year, output_dir):
    path = os.path.join(output_dir, f"ACSDT5Y{year}.{table}-Data.csv")
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerows(rows)
    return path


# A raw census file's rows, without blank lines or the empty last column some website downloads end each row with
def read_census_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        rows = [row for row in csv.reader(f) if row]
    if rows[0][-1] == "":
        rows = [row[:-1] for row in rows]
    return rows

## Download

In [ ]:
# Every ACS block-group file covers the same block groups, so each download must match each of them
raw_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*", "ACSDT5Y*-Data.csv")))
assert raw_paths, "No raw census files in the SCOVI_PROJECT_DIR census/JPL folder"
raw_block_groups = {
    os.path.relpath(raw_path, RAW_DIR): sorted(row[0] for row in read_census_csv(raw_path)[2:])
    for raw_path in raw_paths
}

downloads = {}
for table in TABLES:
    rows = download_table(table, YEAR)
    path = save_census_csv(rows, table, YEAR, OUTPUT_DIR)
    block_groups = sorted(row[0] for row in rows[2:])
    for raw_path, raw_groups in raw_block_groups.items():
        assert raw_groups == block_groups, f"{table}: block groups differ from {raw_path}"
    downloads[table] = rows
    print(f"Saved {len(block_groups)} block groups to {os.path.basename(path)}")

## Compare with the current files

For each downloaded table, finds its current file in the JPL folders, whatever its year, and checks the download has the same column codes, labels and block groups. Same labels means the JPL spreadsheet and the menu still fit the new file.

When `YEAR` is the current file's year, it also compares every cell, which shows whether the download can stand in for one from data.census.gov. `TABLES = ["B25046"]` at 2022 is the best test of that, since it has every kind of missing-value note ("-", "**" and "null").

In [ ]:
for table, rows in downloads.items():
    current_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*", f"ACSDT5Y*.{table}-Data.csv")))
    if not current_paths:
        print(f"{table}: no current file in the JPL folders to compare with")

    new_labels = dict(zip(rows[0], rows[1]))
    new_rows = {row[0]: row for row in rows[2:]}

    for current_path in current_paths:
        current = read_census_csv(current_path)
        current_labels = dict(zip(current[0], current[1]))
        current_rows = {row[0]: row for row in current[2:]}

        print(f"{table}: {os.path.relpath(current_path, RAW_DIR)}")
        print("  Same codes:", current[0] == rows[0])
        print("  Same labels:", current[1] == rows[1])
        print("  Same block groups:", sorted(current_rows) == sorted(new_rows))
        print("  Codes only in the current file:", sorted(set(current_labels) - set(new_labels)))
        print("  Codes only in the download:", sorted(set(new_labels) - set(current_labels)))
        print("  Relabelled:", [
            (code, current_labels[code], new_labels[code])
            for code in current_labels
            if code in new_labels and current_labels[code] != new_labels[code]
        ])

        if os.path.basename(current_path).startswith(f"ACSDT5Y{YEAR}."):
            differences = [
                (geo_id, current[0][i], current_value, new_value)
                for geo_id, row in current_rows.items()
                for i, (current_value, new_value) in enumerate(zip(row, new_rows[geo_id]))
                if current_value != new_value
            ]
            print(f"  {len(differences)} cells differ:", differences[:10])